# Loading packages

In [ ]:
library(Signac)
library(Seurat)
library(rtracklayer)
library(dplyr)
library(Matrix)
library(ggplot2)
library(cowplot)
require(biomaRt)
library(DoubletFinder)
library(RColorBrewer)
set.seed(1234)

# Colour palette

In [ ]:
my_c <- c('#fe864a', '#c9d179', '#95d0fc', '#fbdd7e', '#388004', '#c760ff', '#76cd26', '#ffbacd', '#d6b4fc',
        '#cfaf7b','#8b88f8', '#acc2d9', '#894585', '#82cbb2', '#d99b82', '#fdee73', '#5170d7', '#ffd8b1', 
        '#b7e1a1', '#82a67d', '#b7fffa', '#b790d4')

# Data loading

 This genes are the ones used to perform the aligment on CellRanger using the costumized genome reference. It contains teh CreERT2 gene and e GFP gene.

In [ ]:
annotation <- import('genes.gtf')

In [ ]:
annotation$gene_biotype[1:1780455] = annotation$gene_type[1:1780455]
seqlevelsStyle(annotation) <- "UCSC" #Support for other than Ensembl seqlevel style

# Files and object preparation

The function bellow is an easy way to create each seurat file containing both assays: *RNA* and *ATAC*.

***NOTE:*** Run this cell only the first time you run this notebook.

In [ ]:
Combined_multiome <- function(proj_name, annotation_f) {
    print("-------------------------------------NEW BATCH------------------------------")
    print(proj_name)
    #read the data matrix
    counts_path <- paste(proj_name, "/filtered_feature_bc_matrix.h5", sep="")
    print(counts_path)
    frags_path <- paste(proj_name, "/atac_fragments.tsv.gz", sep="")
    print(frags_path)
    counts <- Read10X_h5(counts_path)
    frags <- frags_path

    #create seurat object with RNA and ATAC
    data_obj <- CreateSeuratObject(counts=counts$`Gene Expression`, assay='RNA', project=proj_name, annotation=annotation_f)
    data_obj[['ATAC']] <- CreateChromatinAssay(counts=counts$Peaks, sep=c(':','-'), fragments=frags, annotation=annotation_f)

    #add metadata
    age <- substring(proj_name,first = 1,last = 3)
    print(age)
    data_obj$Age <- age
    data_obj$Sample <- proj_name


    DefaultAssay(data_obj) <- 'RNA'


    data_obj[['pct_mito']] <- PercentageFeatureSet(data_obj, pattern='^mt')# features=mito_genes)
    data_obj[['pct_hb']] <- PercentageFeatureSet(data_obj, pattern='^Hb[^(p)]')#features=hb_genes)

    ## ATAC

    DefaultAssay(data_obj) <- 'ATAC'

    # Calculate the strength of the nucleosome signal per cell.
    # Computes the ratio of fragments between 147bp and 294bp to fragments<147bp (nucleosome-free)
    print('Calculate Nucleossome signal')
    data_obj <- NucleosomeSignal(data_obj) 
    

    #Calculate TSS enrichment
    print('Calculate TSS enrichment')
    data_obj <- TSSEnrichment(data_obj, fast=FALSE)
    #Save the data processed until now
    save_dir <- paste(proj_name, "_multiome.rds", sep="")
    print(save_dir)
    saveRDS(data_obj, save_dir)

    #delete objects from this iteration from R memory to avoid memory overload
    #delete objects from this iteration from R memory in order to avoid memory overload
    to_del <- c(data_obj, counts, frags, frags_path, counts_path, save_dir)
    for(obj in to_del){
        rm(obj)
    }

}

***NOTE:*** Run the cell bellow only once, this is the first time we run the analysis, since this is the cell that will generate the combined *RNA* and *ATAC* seurat obejct for each sample.
***NOTE:*** Run the cell bellow only once, this is the first time we run the analysis, since this is the cell that will generate the combined *RNA* and *ATAC* seurat obejct for each sample.



In [ ]:
for (pdir in list.dirs('files',recursive = FALSE)){
   proj <- substring(pdir, first=83, last=110)
   Combined_multiome(proj, annotation_f=annotation)
}

Here we will create some files that will be needed for further analysis, in this case to use AMULET for doublets detection in ATAC data.

***NOTE:*** The cell bellow creates the file single_cell.csv for each sample, as requested by the package AMULET. In our case since we don't have cell ids but only the barcodes, the column of the cell id will be filled with the value 0.

In [ ]:
samples <- c('P11_sample1', 'P11_sample2', 'P11_sample3', 'P60_sample1', 'P60_sample2', 'P60_TTR_sample3') 

In [ ]:
# only run once
files_dir <- 'Multiome'
for (sample in samples){
   bc_metrics = read.csv(paste0(files_dir,sample,'/outs/per_barcode_metrics.csv'))
   df = data.frame("barcode" = bc_metrics$barcode, "is__cell_barcode" =  bc_metrics$is_cell)
   write.csv(df,paste0(files_dir, sample,'/single_cell.csv'),row.names = FALSE)
}